In [ ]:
!pip install "unsloth[torch]" trl datasets accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 3.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10

In [ ]:
# Imports
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
from datasets import load_dataset
import torch

# === Paths ===
adapter_model_path = "/content/drive/MyDrive/DAPT+SFT/tinyllama-DAPT-sft-adapter"
merged_model_path = "/content/drive/MyDrive/DAPT+SFT/tinyllama-DAPT-SFT-merged"

# === Load adapter model for DPO training ===
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=adapter_model_path,
    dtype=torch.float16,
    load_in_4bit=True
)
dataset = load_dataset("EliasHossain/dpo-medical-o1-synth", split="train").select(range(6000))
print("Sample row:", dataset[0])


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.1.3 patched 22 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


README.md:   0%|          | 0.00/468 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/882k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/985 [00:00<?, ? examples/s]

Sample row: {'prompt': 'What is the most appropriate immediate therapy for a two-year-old boy presenting to the emergency room with severe anemia (Hb = 2 gm%) and features of congestive heart failure?', 'chosen': "The most appropriate immediate therapy for a two-year-old boy who presents with severe anemia (hemoglobin level of 2 gm%) and features of congestive heart failure is a transfusion of packed red blood cells. This intervention is critical because it rapidly increases the hemoglobin level, thereby enhancing the oxygen-carrying capacity of the blood and alleviating the symptoms of heart failure caused by anemia. Addressing both the anemia and heart failure promptly is essential to stabilize the child's condition and prevent further complications.", 'rejected': "To confirm the cause of the anemia in this 57-year-old woman, the most telling laboratory result would be a low serum ferritin level. This finding is typically indicative of iron deficiency anemia, the most common cause of

In [ ]:
# === DPO Training Config ===
training_args = DPOConfig(
    output_dir="/content/drive/MyDrive/tinyllama-dpo-trained",
    logging_steps=100,
    bf16=False,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=3,   # adjust as needed
    save_strategy="steps",
    save_steps=500,
    learning_rate=2e-5
)

# === Trainer ===
trainer = DPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=dataset
)



Extracting prompt in train dataset (num_proc=6):   0%|          | 0/6000 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/6000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/6000 [00:00<?, ? examples/s]

In [ ]:
# Train
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,000 | Num Epochs = 3 | Total steps = 2,250
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
100,0.806700,0.548665,0.441091,0.535000,0.107574,-240.890884,-318.067444,-1.556550,-1.588044,0,0,0
200,0.125800,0.064158,-4.747864,0.958750,4.812022,-245.424133,-373.476166,-1.395496,-1.374912,No Log,No Log,No Log
300,0.004400,0.039148,-10.750072,1.000000,10.789221,-248.190903,-437.558136,-1.232915,-1.221221,No Log,No Log,No Log
400,0.004100,0.341299,-11.939942,0.998750,12.281241,-252.229996,-442.897339,-1.240761,-1.258891,No Log,No Log,No Log
500,0.001900,0.200807,-14.212827,0.998750,14.413632,-247.820404,-468.496460,-1.279507,-1.279959,No Log,No Log,No Log
600,0.001200,0.180967,-14.383313,1.000000,14.564280,-242.617310,-458.032196,-1.299081,-1.306215,No Log,No Log,No Log
700,0.000800,0.123112,-15.947029,1.000000,16.070137,-245.503235,-482.296448,-1.266009,-1.291077,No Log,No Log,No Log
800,0.000300,0.075765,-16.600628,1.000000,16.676394,-250.248657,-481.499847,-1.280545,-1.297848,No Log,No Log,No Log
900,0.000100,-0.045284,-17.450361,1.000000,17.405079,-250.484360,-500.370239,-1.273715,-1.279864,No Log,No Log,No Log
1000,0.000100,0.103509,-17.697821,1.000000,17.801329,-247.686676,-488.512115,-1.264044,-1.270202,No Log,No Log,No Log


KeyboardInterrupt: 

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/tinyllama-dpo-trained/checkpoint-1000",
    dtype=torch.float16,
    load_in_4bit=False
)

# === Merge LoRA adapters into base model ===
model = model.merge_and_unload()

# === Save final merged model ===
save_path = "/content/drive/MyDrive/tinyllama-dpo-merged-final"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Final merged DPO model saved to: {save_path}")


==((====))==  Unsloth 2026.1.3: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


✅ Final merged DPO model saved to: /content/drive/MyDrive/tinyllama-dpo-merged-final
